# HST Corpus 80m Analysis

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 21.02.2025
- Date of Last Modification: 10.06.2025 (by Sebastian Birk; sb75@sanger.ac.uk)

# 1. Setup

## 1.1 Load Libraries 

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import sys
sys.path.append("../../utils")

In [5]:
import logging
import os
import warnings
import pickle
import random
from datetime import datetime

import anndata as ad
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scib_metrics
import seaborn as sns
import squidpy as sq
from datasets import load_from_disk
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import train_test_split

from app.infer import infer
from app.train import train
from nichejepa.datasets import prepare_dataset
from nichejepa.utils.config import create_params_from_YAML_wandb_config
from nichejepa.utils.distributed import init_distributed
from nichejepa.utils.evaluation import clustering_metrics, classification_metrics

from analysis_utils import *

/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


## 1.2 Load Config/Params
All setup will be done in the config file. Please edit that file to continue.

In [6]:
# Set params
dataset = 'hst_corpus_80m'
tokenizer = 'cell_graph'
# load_timestamp = '29032025_080927_875' # gt_tiny
# load_timestamp = '03042025_224852_256' # gt_small
# load_timestamp = '27032025_163938_061' # gt_base
# load_timestamp = '03042025_224013_784' # gt_large
# load_timestamp = None
load_timestamp = '11062025_183320_055'
model_type = 'test'
read_checkpoint = 'nichejepa-ep10_0.pth.tar' # 'nichejepa-ep1_6.pth.tar'

In [7]:
model_config_folder_path = '/nfs/team361/sb75/nichejepa/configs/model' # '/nfs/team361/sb75/nichejepa/configs/model' # '/lustre/scratch126/cellgen/team361/sb75/nichejepa/configs/model'
artifacts_folder_path = '/nfs/team361/sb75/nichejepa-reproducibility/artifacts' # '/nfs/team361/sb75/nichejepa-reproducibility/artifacts' # '/nfs/team361/sb75/nichejepa-reproducibility/artifacts' # '/lustre/scratch126/cellgen/lotfollahi/sb75/nichejepa-reproducibility/artifacts' # '/nfs/team361/sb75/nichejepa-reproducibility/artifacts'
datasets_folder_path = '/nfs/team361/sb75/DATASETS' # '/lustre/scratch126/cellgen/team361/DATASETS'
token_dict_path = f'{datasets_folder_path}/tokenizer/token_dictionary_homo_sapiens.pkl'

In [8]:
# Set up logger
rank=0
logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO if rank == 0 else logging.ERROR)

In [9]:
# Load config
if not load_timestamp:
    params = create_params_from_YAML_wandb_config(f'{model_config_folder_path}/{dataset}_{model_type}.yaml',logger)
else:
    params = create_params_from_YAML_wandb_config(f'{artifacts_folder_path}/{dataset}/{load_timestamp}/params.yaml',logger)

if read_checkpoint:
    params['state']['read_checkpoint'] = read_checkpoint
if load_timestamp:
    params['state']['load_checkpoint'] = True

params

INFO:root:Loaded parameters from YAML file.


{'data': {'batch_size': 144,
  'dataset_name': 'hst_corpus_80m',
  'n_segments': 11,
  'num_workers': 48,
  'pin_memory': True,
  'precomputed_epoch_n_nonzero_tokens': '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_80m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_epoch_n_nonzero_tokens_30.pkl',
  'precomputed_epoch_splits': '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_80m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_epoch_split_30.pkl',
  'random_state': 0,
  'raw_data_folder_path': '/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/hst_corpus_80m',
  'sample_subset': False,
  'sampling_strategy': None,
  'seq_len_cell': 256,
  'seq_len_neighborhood': 2560,
  'split': 0,
  'stratify': False,
  'token_dict_folder_path': '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/token_dictionary_homo_sapiens.pkl',
  'tokenized_data_folder_path': '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tok

In [10]:
# Define folder path for saving and loading artifacts
if not load_timestamp:
    current_timestamp = (
        datetime.now().strftime("%d%m%Y_%H%M%S") +
        f"_{datetime.now().microsecond // 1000:03d}")
    folder_path = os.path.join(artifacts_folder_path,
                               params['data']['dataset_name'],
                               current_timestamp)
else:
    folder_path = f'{artifacts_folder_path}/{dataset}/{load_timestamp}'
    
folder_path

'/nfs/team361/sb75/nichejepa-reproducibility/artifacts/hst_corpus_80m/11062025_183320_055'

## 1.3 Run Notebook Setup

In [11]:
os.makedirs(f'{folder_path}/figures', exist_ok=True)

In [12]:
warnings.filterwarnings("ignore", category=ResourceWarning)

In [13]:
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

In [14]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

In [15]:
sc.settings.figdir = f'{folder_path}/figures'

sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(6, 4),
    facecolor='white',
    fontsize=10
)

## 1.4 Define Functions

In [16]:
def run_inference(dataset_name,
                  params,
                  read_checkpoint,
                  obs_cols,
                  rerun_inference=False,
                  write_to_disk=True):
    # Load inference results if they exist
    if os.path.exists(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad') and not rerun_inference:
        adata = sc.read_h5ad(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
    else:
        # Load data
        _, _, test_dataset = prepare_dataset(params)
        test_dataset
        
        # Run model inference
        adata = infer(params,
                      test_dataset,
                      load_folder_path=folder_path,
                      dataset_ids=list(set([batch_id.split('_')[0] for batch_id in params['data']['test_batch_ids']])),
                      obs_cols=obs_cols,
                      agg_type='avg',
                      masked_tokens=None,
                      agg_excluded_tokens=None,
                      cell_gene_ids=[],
                      neighborhood_gene_ids=[])

    # Extract batch from cell ID
    adata.obs["batch"] = [cell_id.split('_')[1].replace('batch', '') for cell_id in adata.obs.index.tolist()]
    
    # Remove missing ground truth
    for obs_col in obs_cols:
        adata = adata[~adata.obs[obs_col].isna()]

    if write_to_disk:
        # Save adata
        adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

    return adata

In [17]:
def compute_embedding_similarity(adata,
                                 emb_keys=['cell', 'neighborhood'],
                                 label='niche_type',
                                 write_to_disk=True):
    for emb_key in emb_keys:
        similarity_stats = compute_similarity_stats(
            adata,
            n_samples=None,
            type_key=label,
            embedding_key=f'{emb_key}_emb_layer_{layer}')

        avg_cos_sim = np.mean(similarity_stats["Mean Similarity"])
    
        cosine_df = pd.DataFrame({
            'metric': [f'avg_{label}_cos_sim'],
            'value': [avg_cos_sim]
        })
        print(f'Average {label} cosine similarity is {avg_cos_sim}.')
        cosine_df.to_csv(f'{folder_path}/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_avg_{label}_cos_sim.csv',
                         index=False)
        
        plot_similarity_heatmaps(
            similarity_stats,
            save_file_path=f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{emb_key}_emb_{label}_cos_sim.svg')

In [18]:
def compute_neighborhood_umap(adata,
                              write_to_disk=True):
    if 'neighborhood_connectivities' not in adata.obsp.keys():
        # Compute UMAP representation for (aggregated) neighborhood gene embeddings
        sc.pp.neighbors(adata,
                        n_neighbors=15,
                        use_rep=f'neighborhood_emb_layer_{layer}',
                        key_added='neighborhood')
    if 'neighborhood_umap' not in adata.obsm.keys():
        sc.tl.umap(adata,
                   neighbors_key='neighborhood',
                   key_added='neighborhood_umap')
    
        if write_to_disk:
            # Save adata
            adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
    
    if 'cell_connectivities' not in adata.obsp.keys():
        # Compute UMAP representation for (aggregated) cell gene embeddings
        sc.pp.neighbors(adata,
                        n_neighbors=15,
                        use_rep=f'cell_emb_layer_{layer}',
                        key_added='cell')
    if 'cell_umap' not in adata.obsm.keys():
        sc.tl.umap(adata,
                   neighbors_key='cell',
                   key_added='cell_umap')

        if write_to_disk:
            # Save adata
            adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

    return adata

In [19]:
def compute_latent_clusters(adata,
                            emb_keys=['cell', 'neighborhood'],
                            obs_col='niche_type',
                            latent_leiden_resolution_cell=1.0,
                            latent_leiden_resolution_neighborhood=0.4,
                            write_to_disk=True):
    for emb_key in emb_keys:
        latent_leiden_resolutions = []
        if emb_key == 'cell':
            latent_leiden_resolution = latent_leiden_resolution_cell
        elif emb_key == 'neighborhood':
            latent_leiden_resolution = latent_leiden_resolution_neighborhood
        adata.obsm['X_umap'] = adata.obsm[f'{emb_key}_umap']
        # Perform clustering until number of ground truth niches is matched
        i = 0
        n_unique_vals = len(adata.obs[obs_col].value_counts() < 100)
        print(f'Target number of unique values: {n_unique_vals}.')
        n_latent_clusters = 0
        while (n_latent_clusters != n_unique_vals):
            if latent_leiden_resolution in latent_leiden_resolutions:
                latent_leiden_resolution += random.uniform(0.0001, 0.001)
            latent_leiden_resolutions.append(latent_leiden_resolution)
            print(f'Leiden resolution: {latent_leiden_resolution}.')
            latent_cluster_key = f'{emb_key}_emb_leiden_res{str(latent_leiden_resolution).replace(".", "_")}'
            i += 1
            sc.tl.leiden(adata,
                         neighbors_key=emb_key,
                         key_added=latent_cluster_key,
                         resolution=latent_leiden_resolution)
        
            n_latent_clusters = len(adata.obs[latent_cluster_key].value_counts() < 100)
            print(f"Number of latent clusters {n_latent_clusters}.")
            if n_latent_clusters == n_unique_vals:
                break
            elif n_latent_clusters < (n_unique_vals - 8):
                latent_leiden_resolution += 0.1
            elif n_latent_clusters < (n_unique_vals - 5):
                latent_leiden_resolution += 0.05
            elif n_latent_clusters < (n_unique_vals - 3):
                latent_leiden_resolution += 0.02
            elif n_latent_clusters < n_unique_vals:
                latent_leiden_resolution += 0.001
            elif n_latent_clusters > (n_unique_vals + 8):
                latent_leiden_resolution -= 0.1
            elif n_latent_clusters > (n_unique_vals + 5):
                latent_leiden_resolution -= 0.05
            elif n_latent_clusters > (n_unique_vals + 3):
                latent_leiden_resolution -= 0.02
            else:
                latent_leiden_resolution -= 0.001
        
        # Compute metrics and store to disk
        labels = adata.obs[obs_col]
        labels_pred = adata.obs[latent_cluster_key]
        nmi = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
        ari = adjusted_rand_score(labels, labels_pred)
        print(f'NMI is {nmi}. ARI is {ari}.')
        metrics_df = pd.DataFrame({
            'metric': ['NMI', 'ARI'],
            'value': [nmi, ari]
        })
        metrics_df.to_csv(f'{folder_path}/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_metrics.csv',
                          index=False)
        
        # Plot UMAPs
        os.makedirs(f'{folder_path}/figures', exist_ok=True)
        labels = ['batch', obs_col, latent_cluster_key]
        # Plot (aggregated) neighborhood gene embeddings, colored by labels
        for label in labels:
            if len(np.unique(adata.obs[label])) > 1:
                sc.pl.umap(adata,
                           neighbors_key=emb_key,
                           title=(emb_key + "_emb_" + label if emb_key not in label else label),
                           color=label,
                           palette=(custom_palette if label == latent_cluster_key else None),
                           show=False)
                plt.savefig(f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}{"_" + emb_key + "_emb" if emb_key not in label else ""}_{label}_umap.svg',
                            dpi=300,
                            bbox_inches='tight')
        
        # Plot niches in tissue
        batch_ids = adata.obs['batch'].unique().tolist()
        for i, batch_id in enumerate(batch_ids):
            adata_batch = adata[adata.obs['batch'] == batch_id]
            sc.pl.spatial(adata_batch,
                          color=latent_cluster_key,
                          palette=custom_palette,
                          spot_size=spot_sizes[i],
                          save=f'{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg')
            os.rename(f'{folder_path}/figures/show{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg',
                      f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg'.replace("show", ""))
    
        if write_to_disk:
            # Save adata
            adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
    
    return adata

# 2. Analysis

In [20]:
# Set inference params
layer = params['meta']['enc_depth']
exclude_unshared_tokens = False # True, False
agg_type = 'avg' # 'avg', 'weighted_avg', 'cls'
params['data']['batch_size'] = 128

In [21]:
params['data']['precomputed_n_nonzero_tokens'] = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_80m_32_None_None_None_None_None_shifted_log_knn_10_n_nonzero_tokens_train.pkl'

## 2.2 Full Corpus

### 2.2.1 Run Model Inference

In [ ]:
# Load token dict
with open(token_dict_path, 'rb') as f:
    token_dict = pickle.load(f)

token_dict_reverse =  {value: key for key, value in token_dict.items()}

spv_dict = {
    k: v for k, v in token_dict.items() if k.startswith('spv_')}
spv_start_idx = min(spv_dict.values())
max_cls_tokens = sum(1 for key in token_dict if "cls" in key)
spv_idx_subtract = spv_start_idx - 2 - max_cls_tokens

In [ ]:
# Load dataset
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = False
params['data']['val_batch_ids'] = False
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

dataset, _, _ = prepare_dataset(params)
dataset

In [ ]:
# Set inference params
layer = params['meta']['enc_depth']
exclude_unshared_tokens = False # True, False
agg_type = 'avg' # 'avg', 'weighted_avg', 'cls'
#latent_leiden_resolution = 0.5 # 0.1, 0.5
#latent_cluster_key = f'leiden_res{str(latent_leiden_resolution).replace(".", "_")}'

### 2.2.3 Perform Cross Tissue Analysis on 5k Gene Panel

In [22]:
# Load token dict
with open(token_dict_path, 'rb') as f:
    token_dict = pickle.load(f)

token_dict_reverse =  {value: key for key, value in token_dict.items()}

spv_dict = {
    k: v for k, v in token_dict.items() if k.startswith('spv_')}
spv_start_idx = min(spv_dict.values())
max_cls_tokens = sum(1 for key in token_dict if "cls" in key)
spv_idx_subtract = spv_start_idx - 2 - max_cls_tokens

In [23]:
# Load dataset
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = False
params['data']['val_batch_ids'] = False
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

dataset, _, _ = prepare_dataset(params)
dataset

Dataset({
    features: ['cell_id', 'batch_token', 'gene_panel_token', 'assay_token', 'species_token', 'tissue_token', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'seg_tokens', 'n_nonzero_tokens'],
    num_rows: 80172337
})

In [24]:
from collections import defaultdict
from itertools import chain

from datasets import concatenate_datasets

from app.infer import get_gene_embed

In [25]:
model_folder_path = '/nfs/team361/sb75/nichejepa-reproducibility/artifacts/models/11062025_183320_055'

In [26]:
emb_layer = None
top_k = None
batch_size = 128
pin_memory = False
num_workers = 12

In [27]:
##### Filter dataset to keep only 5k gene panel cells #####
# Get gene panel values for 5k gene panel
min_gene_panel_size = 4948
max_gene_panel_size = 4949

# Perform index subtraction as in tokenizer
max_cls_tokens = 100
spv_dict = {
    k: v for k, v in token_dict.items() if k.startswith('spv_')}
spv_start_idx = min(spv_dict.values())
spv_idx_subtract = spv_start_idx - 2 - max_cls_tokens

keep_gene_panel_values = torch.tensor([
    v - spv_idx_subtract for k,v in token_dict.items() if ('spv_gene_panel' in k) and (int(k.split('panel')[1]) >= min_gene_panel_size) and (int(k.split('panel')[1]) <= max_gene_panel_size)])

# Filter dataset
dataset.set_format(type=None) # for extreme speed-up

def keep_row(x):
    return torch.isin(torch.tensor(x["gene_panel_value"]), keep_gene_panel_values)

filtered_dataset = dataset.filter(
    keep_row,
    batched=True,
    batch_size=10000, 
    num_proc=num_workers)

######################################################################

##### Subsample dataset and only keep 100k genes per tissue #####
n_per_tissue = 100_000
rng_seed = 42

# Step 1: Shuffle the full dataset once
shuffled_dataset = filtered_dataset.shuffle(seed=rng_seed)

# Step 2: Extract the tissue_value column (this is fast and avoids full row loading)
tissue_values = shuffled_dataset["tissue_value"]

# Step 3: Collect sample indices efficiently
tissue_counts = defaultdict(int)
tissue_indices = defaultdict(list)

for idx, tissue in enumerate(tissue_values):
    tissue = tissue[0]

    if tissue_counts[tissue] < n_per_tissue:
        tissue_indices[tissue].append(idx)
        tissue_counts[tissue] += 1

# Step 4: Build final dataset with select
final_samples = [
    shuffled_dataset.select(indices) for indices in tissue_indices.values()
]
final_dataset = concatenate_datasets(final_samples)
columns = list(final_dataset.features.keys())
columns.remove("cell_id")
final_dataset.set_format(
    type="torch",
    columns=columns,
    output_all_columns=True)

# Step 5: Write to disk
#final_dataset.save_to_disk(
#    "/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_80m/hst_corpus_80m_32_None_None_None_None_None_shifted_log_knn_10_5k_subsample.dataset")
######################################################################

In [50]:
import random

adata_temp = sc.read_h5ad('/nfs/team361/sb75/DATASETS/silver/xhe1008-1-20b_1p/section020_SOB26.h5ad')
random.seed(42)
random_genes = random.sample(adata_temp.var.index.tolist(), 14)

In [51]:
random_genes

['COIL',
 'ANO1',
 'KAT2B',
 'HSPD1',
 'GRM5',
 'DEK',
 'CIP2A',
 'TMEM126A',
 'CDC34',
 'XPR1',
 'PROM1',
 'ARF6',
 'APOL1',
 'CEACAM8']

In [52]:
# Specify genes
for gene_category in ['random']: # ['pan_tcells', 'pan_tams', 'tissue_specific', 'random']:
    pan_tcells_genes =[
        'CD3D', # 'ENSG00000167286'
        'CD3E', # 'ENSG00000198851'
        'CD3G', # 'ENSG00000160654'
        'CD4', # 'ENSG00000010610'
        'CD8A', # 'ENSG00000153563'
        'CD2', # 'ENSG00000116824'
        'CD5', # 'ENSG00000110448'
        'CD7', # 'ENSG00000173762'
        'TRAC', # ''
        'TRBC1', # ''
        'TRBC2', # ''
        'IL7R', # 'ENSG00000168685'
        'TCF7', # 'ENSG00000081059'
        'SELL', # 'ENSG00000188404'
        'CCR7' # 'ENSG00000126353'
    ]
    pan_tams_genes = [
        'CD68', # ENSG00000129226
        'C1QA', # ENSG00000173372
        'ITGA4', # ENSG00000115232
        'P2RY12', # ENSG00000169313
        'CD163', # ENSG00000177575
        'CD206', # ''
        'CSF1R', # ENSG00000182578
        'F13A1', # ENSG00000124491
        'APOE', # ENSG00000130203
        'LYZ', # ENSG00000090382
        'AIF1', # ENSG00000204472
        'TREM2', # ENSG00000095970
        'MS4A6A' # ENSG00000110077
    ]
    tissue_specific_genes = [
        'ALB',
        'FGB',
        'MBP',
        'MOBP',
        'GFAP',
        'SFTPC',
        'SFTPA1',
        'KRT1',
        'AQP1',
        'UMOD']
    random_genes = [
        'COIL',
        'ANO1',
        'KAT2B',
        'HSPD1',
        'GRM5',
        'DEK',
        'CIP2A',
        'TMEM126A',
        'CDC34',
        'XPR1',
        'PROM1',
        'ARF6',
        'APOL1',
        'CEACAM8']
    
    if gene_category == 'pan_tcells':
        genes = pan_tcells_genes
    elif gene_category == 'pan_tams':
        genes = pan_tams_genes
    elif gene_category == 'tissue_specific':
        genes = tissue_specific_genes
    elif gene_category == 'random':
        genes = random_genes
    
    # Get ensembl IDs
    gene_name_to_ensembl_id_dict_path = '/nfs/team361/sb75/DATASETS/genes/homo_sapiens_gene_name_to_ensembl_id_dict.pkl'
    with open(gene_name_to_ensembl_id_dict_path, 'rb') as f:
        gene_name_to_ensembl_id_dict = pickle.load(f)
    
    ensembl_ids = [gene_name_to_ensembl_id_dict.get(gene, '') for gene in genes]
    cell_gene_ensembl_id = [gene for gene in ensembl_ids if gene != '']
    neighborhood_gene_ensembl_id = [gene for gene in ensembl_ids if gene != '']
    
    # Get gene embeddings
    output_gene_embed = get_gene_embed(
        dataset=final_dataset,
        model_folder_path=model_folder_path,
        emb_layer=emb_layer,
        batch_size=batch_size,
        pin_memory=pin_memory,
        num_workers=num_workers,
        cell_gene_ensembl_id=cell_gene_ensembl_id,
        neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
    )
    
    # Save output_gene_embed dict
    with open(f'{folder_path}/output_gene_embed_{gene_category}_{read_checkpoint.split("-")[1].split(".")[0]}.pkl', 'wb') as f:
        pickle.dump(output_gene_embed, f)

STEP 1: LOADING CONFIG...
STEP 2: GETTING GENE EMBEDDINGS


INFO:root:EncoderMultiMaskWrapper(
  (backbone): GeneTransformerCombinedEncoder(
    (token_embed): Embedding(23407, 384, padding_idx=0)
    (seg_embed): Embedding(12, 384, padding_idx=0)
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (norm): LayerNorm((384,), eps=1e-06, elementwise_affi

['encoder', 'predictor', 'target_encoder', 'opt', 'scaler', 'epoch', 'zero_epoch_tracking', 'loss', 'batch_size', 'world_size', 'lr', 'iter_number']


0it [00:00, ?it/s]/nfs/team361/sb75/nichejepa/src/app/infer.py:1450: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/software/cellgen/team361/sb75/envs/nichejepa/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
5469it [2:59:43,  1.97s/it]


In [53]:
2+3

5

In [54]:
gene_category = 'random'

In [55]:
f'{folder_path}/output_gene_embed_{gene_category}_{read_checkpoint.split("-")[1].split(".")[0]}.pkl'

'/nfs/team361/sb75/nichejepa-reproducibility/artifacts/hst_corpus_80m/11062025_183320_055/output_gene_embed_random_ep10_0.pkl'

In [56]:
# Load output_gene_embed dict
with open(f'{folder_path}/output_gene_embed_{gene_category}_{read_checkpoint.split("-")[1].split(".")[0]}.pkl', 'rb') as f:
    output_gene_embed = pickle.load(f)

In [57]:
ensembl_id_to_gene_name_dict = {
    v: k for k, v in gene_name_to_ensembl_id_dict.items()}

label = 'tissue'

for key in output_gene_embed.keys():
    if output_gene_embed[key].sum() != 0:
        # 1. Filter nonzero rows
        nonzero_mask = ~(output_gene_embed[key] == 0).all(axis=1)
        filtered_gene_embed = output_gene_embed[key][nonzero_mask]

        # 2. Prepare labels
        filtered_tissues = np.array([
            token_dict_reverse[tissue_value.item() + spv_idx_subtract].split('spv_')[1]
            for tissue_value in final_dataset['tissue_value']
        ])[nonzero_mask]

        # 3. Create AnnData
        adata = ad.AnnData(filtered_gene_embed)
        adata.obs['tissue'] = filtered_tissues

        # 4. Neighbors + UMAP
        sc.pp.neighbors(adata, use_rep='X')
        sc.tl.umap(adata)
        sc.pl.umap(adata,
                   size=3,
                   color=label,
                   show=False)
        plt.title(f'{key.split("gene")[0] + ensembl_id_to_gene_name_dict.get(key.split("gene")[1])} by {label} (UMAP)', fontsize=14)
        plt.savefig(
            f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/figures/{key.split("gene")[0] + ensembl_id_to_gene_name_dict.get(key.split("gene")[1])}_umap_{label}.png',
            dpi=300,
            bbox_inches='tight')
        plt.clf()

        # 5. PCA
        sc.tl.pca(adata, use_highly_variable=False, svd_solver='arpack')
        sc.pl.pca(adata,
                  color=label,
                  size=3,
                  show=False)
        plt.title(f'{key.split("gene")[0] + ensembl_id_to_gene_name_dict.get(key.split("gene")[1])} by {label} (PCA)', fontsize=14)
        plt.savefig(
            f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/figures/{key.split("gene")[0] + ensembl_id_to_gene_name_dict.get(key.split("gene")[1])}_pca_{label}.png',
            dpi=300,
            bbox_inches='tight')
        plt.clf()

... storing 'tissue' as categorical
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(adata, mask_var, use_highly_variable)
... storing 'tissue' as categorical
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(adata, mask_var, use_highly_variable)
... storing 'tissue' as categorical
/nfs/users/nfs_s/sb75/.venvs/nichejepa

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

... (47 outputs left out)

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

In [58]:
# Combine all non-zero gene embeddings and collect corresponding labels
for emb_key in ['cell_emb', 'neighborhood_emb']:
    
    filtered_gene_embeds = {}
    tissues = []
    cell_ids = []
    
    for key in output_gene_embed.keys():
        if (output_gene_embed[key].sum() != 0) and (emb_key in key):
            # 1. Filter nonzero rows
            nonzero_mask = ~(output_gene_embed[key] == 0).all(axis=1)
            filtered_gene_embed = output_gene_embed[key][nonzero_mask]
            filtered_gene_embeds[key] = filtered_gene_embed
            
            # 2. Prepare labels
            filtered_tissues = list(np.array([
                token_dict_reverse[tissue_value.item() + spv_idx_subtract].split('spv_')[1]
                for tissue_value in final_dataset['tissue_value']])[nonzero_mask])
            filtered_cell_ids = list(np.array(final_dataset['cell_id'])[nonzero_mask])
            tissues.extend(filtered_tissues)
            cell_ids.extend(filtered_cell_ids)
    
            #print(len(filtered_tissues))
            #print(len(filtered_cell_ids))
            #print(filtered_gene_embed.shape)
    
    concatenated_gene_embed = np.concatenate(
        [v for k, v in filtered_gene_embeds.items() if emb_key in k],
        axis=0)
    
    # 3. Create AnnData
    adata = ad.AnnData(concatenated_gene_embed)
    adata.obs['cell_id'] = cell_ids
    adata.obs['tissue'] = tissues
    adata.obs['ensembl_id'] = list(
        chain.from_iterable([[k] * len(filtered_gene_embeds[k]) for k, v in filtered_gene_embeds.items() if emb_key in k]))
    adata.obs['ensembl_id'] = adata.obs['ensembl_id'].apply(lambda x: x.split('gene')[1])
    adata.obs['gene_name'] = adata.obs['ensembl_id'].apply(lambda x: ensembl_id_to_gene_name_dict.get(x))
    
    for label in ['tissue', 'gene_name']:
        # 4. Neighbors + UMAP
        sc.pp.neighbors(adata, use_rep='X')
        sc.tl.umap(adata)
        sc.pl.umap(adata,
                   size=3,
                   color=label,
                   show=False)
        plt.title(f'{emb_key}_{gene_category} by {label} (UMAP)', fontsize=14)
        plt.savefig(
            f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/figures/{emb_key}_{gene_category}_umap_{label}.png',
            dpi=300,
            bbox_inches='tight')
        plt.clf()
        
        # 5. PCA
        sc.tl.pca(adata, use_highly_variable=False, svd_solver='arpack')
        sc.pl.pca(adata,
                  color=label,
                  size=3,
                  show=False)
        plt.title(f'{emb_key}_{gene_category} by {label} (PCA)', fontsize=14)
        plt.savefig(
            f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/figures/{emb_key}_{gene_category}_pca_{label}.png',
            dpi=300,
            bbox_inches='tight')
        plt.clf()

... storing 'cell_id' as categorical
... storing 'tissue' as categorical
... storing 'ensembl_id' as categorical
... storing 'gene_name' as categorical
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(adata, mask_var, use_highly_variable)
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(adata, mask_var, use_highly_vari

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

In [64]:
# Plot gene program embeddings in shared embedding space
for emb_key in ['cell_emb', 'neighborhood_emb']:
    
    filtered_mean_gene_embeds = {}
    tissues = []
    cell_ids = []
    programs = []
    
    for i, gene_category in enumerate(['pan_tcells', 'pan_tams', 'random']):
        
        # Load output_gene_embed dict for gene category
        with open(f'{folder_path}/output_gene_embed_{gene_category}_{read_checkpoint.split("-")[1].split(".")[0]}.pkl', 'rb') as f:
            output_gene_embed = pickle.load(f)
        
        gene_emb = [v for k, v in output_gene_embed.items() if emb_key in k]
        
        # Stack into shape: (12, 700000, 384)
        stacked = np.stack(gene_emb, axis=0)
        # Mask of nonzero rows: shape (12, 700000)
        nonzero_mask = ~(stacked == 0).all(axis=2)
        # Count of valid arrays per row: shape (700000,)
        valid_counts = nonzero_mask.sum(axis=0)
        # To avoid divide-by-zero later
        valid_counts = np.where(valid_counts == 0, 1, valid_counts)
        # Zero out fully-zero rows before summing
        masked = stacked * nonzero_mask[:, :, np.newaxis]  # shape: (12, 700000, 384)
        # Sum across arrays and normalize
        sum_embeddings = masked.sum(axis=0)                 # shape: (700000, 384)
        mean_gene_emb = sum_embeddings / valid_counts[:, np.newaxis]  # shape: (700000, 384)

        # Remove zero rows in mean_gene_emb
        nonzero_mask = ~(mean_gene_emb == 0).all(axis=1)
        filtered_mean_gene_emb = mean_gene_emb[nonzero_mask]
        filtered_mean_gene_embeds[gene_category] = filtered_mean_gene_emb

        # Prepare labels
        filtered_tissues = list(np.array([
            token_dict_reverse[tissue_value.item() + spv_idx_subtract].split('spv_')[1]
            for tissue_value in final_dataset['tissue_value']])[nonzero_mask])
        filtered_cell_ids = list(np.array(final_dataset['cell_id'])[nonzero_mask])
        tissues.extend(filtered_tissues)
        cell_ids.extend(filtered_cell_ids)
        programs.extend([gene_category] * len(filtered_cell_ids))

    concatenated_mean_gene_embed = np.concatenate(
        [v for k, v in filtered_mean_gene_embeds.items()],
        axis=0)
    
    # Create AnnData
    adata = ad.AnnData(concatenated_mean_gene_embed)
    adata.obs['cell_id'] = cell_ids
    adata.obs['tissue'] = tissues
    adata.obs['program'] = programs
    
    for label in ['tissue', 'program']:
        # 4. Neighbors + UMAP
        sc.pp.neighbors(adata, use_rep='X')
        sc.tl.umap(adata)
        sc.pl.umap(adata,
                   size=3,
                   color=label,
                   show=False)
        plt.title(f'{emb_key}_all_programs by {label} (UMAP)', fontsize=14)
        plt.savefig(
            f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/figures/{emb_key}_all_programs_umap_{label}.png',
            dpi=300,
            bbox_inches='tight')
        plt.clf()
        
        # 5. PCA
        sc.tl.pca(adata, use_highly_variable=False, svd_solver='arpack')
        sc.pl.pca(adata,
                  color=label,
                  size=3,
                  show=False)
        plt.title(f'{emb_key}_all_programs by {label} (PCA)', fontsize=14)
        plt.savefig(
            f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/figures/{emb_key}_all_programs_pca_{label}.png',
            dpi=300,
            bbox_inches='tight')
        plt.clf()

... storing 'cell_id' as categorical
... storing 'tissue' as categorical
... storing 'program' as categorical
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(adata, mask_var, use_highly_variable)
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(adata, mask_var, use_highly_variable)
... storing 'cell_id' as categorical

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

In [66]:
# Plot gene embeddings in shared embedding space
for emb_key in ['neighborhood_emb']:
    
    filtered_gene_embeds = {}
    tissues = []
    cell_ids = []
    programs = []
    
    for i, gene_category in enumerate(['pan_tcells', 'pan_tams', 'random']):

        # Load output_gene_embed dict for gene category
        with open(f'{folder_path}/output_gene_embed_{gene_category}_{read_checkpoint.split("-")[1].split(".")[0]}.pkl', 'rb') as f:
            output_gene_embed = pickle.load(f)
    
        for key in output_gene_embed.keys():
            if (output_gene_embed[key].sum() != 0) and (emb_key in key):
                # 1. Filter nonzero rows
                nonzero_mask = ~(output_gene_embed[key] == 0).all(axis=1)
                filtered_gene_embed = output_gene_embed[key][nonzero_mask]
                filtered_gene_embeds[key] = filtered_gene_embed
                
                # 2. Prepare labels
                filtered_tissues = list(np.array([
                    token_dict_reverse[tissue_value.item() + spv_idx_subtract].split('spv_')[1]
                    for tissue_value in final_dataset['tissue_value']])[nonzero_mask])
                filtered_cell_ids = list(np.array(final_dataset['cell_id'])[nonzero_mask])
                tissues.extend(filtered_tissues)
                cell_ids.extend(filtered_cell_ids)
                programs.extend([gene_category] * len(filtered_cell_ids))
        
    concatenated_gene_embed = np.concatenate(
        [v for k, v in filtered_gene_embeds.items() if emb_key in k],
        axis=0)
    
    # 3. Create AnnData
    adata = ad.AnnData(concatenated_gene_embed)
    adata.obs['cell_id'] = cell_ids
    adata.obs['tissue'] = tissues
    adata.obs['program'] = programs
    adata.obs['ensembl_id'] = list(
        chain.from_iterable([[k] * len(filtered_gene_embeds[k]) for k, v in filtered_gene_embeds.items() if emb_key in k]))
    adata.obs['ensembl_id'] = adata.obs['ensembl_id'].apply(lambda x: x.split('gene')[1])
    adata.obs['gene_name'] = adata.obs['ensembl_id'].apply(lambda x: ensembl_id_to_gene_name_dict.get(x))
    
    for label in ['tissue', 'program', 'gene_name']:
        # 4. Neighbors + UMAP
        sc.pp.neighbors(adata, use_rep='X')
        sc.tl.umap(adata)
        sc.pl.umap(adata,
                   size=3,
                   color=label,
                   show=False)
        plt.title(f'{emb_key}_all_genes by {label} (UMAP)', fontsize=14)
        plt.savefig(
            f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/figures/{emb_key}_all_genes_umap_{label}.png',
            dpi=300,
            bbox_inches='tight')
        plt.clf()
        
        # 5. PCA
        sc.tl.pca(adata, use_highly_variable=False, svd_solver='arpack')
        sc.pl.pca(adata,
                  color=label,
                  size=3,
                  show=False)
        plt.title(f'{emb_key}_all_genes by {label} (PCA)', fontsize=14)
        plt.savefig(
            f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/figures/{emb_key}_all_genes_pca_{label}.png',
            dpi=300,
            bbox_inches='tight')
        plt.clf()

/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/umap/spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(
... storing 'cell_id' as categorical
... storing 'tissue' as categorical
... storing 'program' as categorical
... storing 'ensembl_id' as categorical
... storing 'gene_name' as categorical
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(adata, mask_var, use_highly_variable)
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/umap/s

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

<Figure size 1800x1200 with 0 Axes>

In [67]:
2+3

5

In [ ]:
label = 'tissue'

for key in output_gene_embed.keys():
    if output_gene_embed[key].sum() != 0:
        # 1. Filter nonzero rows
        nonzero_mask = ~(output_gene_embed[key] == 0).all(axis=1)
        filtered_gene_embed = output_gene_embed[key][nonzero_mask]

        # 2. Prepare labels
        filtered_tissues = np.array([
            token_dict_reverse[tissue_value.item() + spv_idx_subtract].split('spv_')[1]
            for tissue_value in final_dataset['tissue_value']
        ])[nonzero_mask]

        # 3. Create AnnData
        adata = ad.AnnData(filtered_gene_embed)
        adata.obs['tissue'] = filtered_tissues

        # 4. Neighbors + UMAP
        sc.pp.neighbors(adata, use_rep='X')
        sc.tl.umap(adata)
        sc.pl.umap(adata,
                   size=3,
                   color=label,
                   show=False)
        plt.title(f'{key} by {label} (UMAP)', fontsize=14)
        plt.savefig(
            f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/figures/{key}_umap.png',
            dpi=300,
            bbox_inches='tight')
        plt.clf()

        # 5. PCA
        sc.tl.pca(adata, use_highly_variable=False, svd_solver='arpack')
        sc.pl.pca(adata,
                  color=label,
                  size=3,
                  show=False)
        plt.title(f'{key} by {label} (PCA)', fontsize=14)
        plt.savefig(
            f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/figures/{key}_pca.png',
            dpi=300,
            bbox_inches='tight')
        plt.clf()

In [ ]:
# Create UMAP and PCA plots for tan PAMs gene embeddings
pan_tams_genes = ['CD68', 'C1QA', 'ITGA4', 'P2RY12', 'CD163', 'CD206', 'CSF1R', 'F13A1', 'APOE', 'LYZ', 'AIF1', 'TREM2', 'MS4A6A']
pan_tams_ensembl_ids = [gene_name_to_ensembl_id_dict.get(gene, '') for gene in pan_tams_genes]

# Specify ensembl IDs
cell_gene_ensembl_id = [
    'ENSG00000129226',
    'ENSG00000173372',
    'ENSG00000115232',
    'ENSG00000169313',
    'ENSG00000177575',
    'ENSG00000182578',
    'ENSG00000124491',
    'ENSG00000130203',
    'ENSG00000090382',
    'ENSG00000204472',
    'ENSG00000095970',
    'ENSG00000110077']
neighborhood_gene_ensembl_id = [
    'ENSG00000129226',
    'ENSG00000173372',
    'ENSG00000115232',
    'ENSG00000169313',
    'ENSG00000177575',
    'ENSG00000182578',
    'ENSG00000124491',
    'ENSG00000130203',
    'ENSG00000090382',
    'ENSG00000204472',
    'ENSG00000095970',
    'ENSG00000110077']

# Get gene embeddings
output_gene_embed = get_gene_embed(
    dataset=final_dataset,
    model_folder_path=model_folder_path,
    emb_layer=emb_layer,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers,
    cell_gene_ensembl_id=cell_gene_ensembl_id,
    neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
)

# Save output_gene_embed dict
with open(f'{folder_path}/output_gene_embed_pan_tams_{read_checkpoint.split("-")[1].split(".")[0]}.pkl', 'wb') as f:
    pickle.dump(output_gene_embed, f)

label = 'tissue'

for key in output_gene_embed.keys():
    if output_gene_embed[key].sum() != 0:
        # 1. Filter nonzero rows
        nonzero_mask = ~(output_gene_embed[key] == 0).all(axis=1)
        filtered_gene_embed = output_gene_embed[key][nonzero_mask]

        # 2. Prepare labels
        filtered_tissues = np.array([
            token_dict_reverse[tissue_value.item() + spv_idx_subtract].split('spv_')[1]
            for tissue_value in final_dataset['tissue_value']
        ])[nonzero_mask]

        # 3. Create AnnData
        adata = ad.AnnData(filtered_gene_embed)
        adata.obs['tissue'] = filtered_tissues

        # 4. Neighbors + UMAP
        sc.pp.neighbors(adata, use_rep='X')
        sc.tl.umap(adata)
        sc.pl.umap(adata,
                   size=3,
                   color=label,
                   show=False)
        plt.title(f'{key} by {label} (UMAP)', fontsize=14)
        plt.savefig(
            f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/figures/{key}_umap.png',
            dpi=300,
            bbox_inches='tight')
        plt.clf()

        # 5. PCA
        sc.tl.pca(adata, use_highly_variable=False, svd_solver='arpack')
        sc.pl.pca(adata,
                  color=label,
                  size=3,
                  show=False)
        plt.title(f'{key} by {label} (PCA)', fontsize=14)
        plt.savefig(
            f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/figures/{key}_pca.png',
            dpi=300,
            bbox_inches='tight')
        plt.clf()

In [ ]:
output_gene_embed

In [ ]:
# Combine all non-zero gene embeddings and collect corresponding labels
for emb_key in ['cell_emb', 'neighborhood_emb']:
    
    filtered_gene_embeds = {}
    tissues = []
    cell_ids = []
    
    for key in output_gene_embed.keys():
        if (output_gene_embed[key].sum() != 0) and (emb_key in key):
            # 1. Filter nonzero rows
            nonzero_mask = ~(output_gene_embed[key] == 0).all(axis=1)
            filtered_gene_embed = output_gene_embed[key][nonzero_mask]
            filtered_gene_embeds[key] = filtered_gene_embed
            
            # 2. Prepare labels
            filtered_tissues = list(np.array([
                token_dict_reverse[tissue_value.item() + spv_idx_subtract].split('spv_')[1]
                for tissue_value in final_dataset['tissue_value']])[nonzero_mask])
            filtered_cell_ids = list(np.array(final_dataset['cell_id'])[nonzero_mask])
            tissues.extend(filtered_tissues)
            cell_ids.extend(filtered_cell_ids)
    
            #print(len(filtered_tissues))
            #print(len(filtered_cell_ids))
            #print(filtered_gene_embed.shape)
    
    concatenated_gene_embed = np.concatenate(
        [v for k, v in filtered_gene_embeds.items() if emb_key in k],
        axis=0)
    
    # 3. Create AnnData
    adata = ad.AnnData(concatenated_gene_embed)
    adata.obs['cell_id'] = cell_ids
    adata.obs['tissue'] = tissues
    adata.obs['ensembl_id'] = list(
        chain.from_iterable([[k] * len(filtered_gene_embeds[k]) for k, v in filtered_gene_embeds.items() if emb_key in k]))
    
    for label in ['tissue', 'ensembl_id']:
        # 4. Neighbors + UMAP
        sc.pp.neighbors(adata, use_rep='X')
        sc.tl.umap(adata)
        sc.pl.umap(adata,
                   size=3,
                   color=label,
                   show=False)
        plt.title(f'{gene_category} {emb_key} by {label} (UMAP)', fontsize=14)
        plt.savefig(
            f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/figures/{gene_category}_{emb_key}_{label}_umap.png',
            dpi=300,
            bbox_inches='tight')
        plt.clf()
        
        # 5. PCA
        sc.tl.pca(adata, use_highly_variable=False, svd_solver='arpack')
        sc.pl.pca(adata,
                  color=label,
                  size=3,
                  show=False)
        plt.title(f'{gene_category} {emb_key} by {label} (PCA)', fontsize=14)
        plt.savefig(
            f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/figures/{gene_category}_{emb_key}_{label}_pca.png',
            dpi=300,
            bbox_inches='tight')
        plt.clf()

In [ ]:
tissue_specific_genes = 

In [ ]:
len(cell_ids)

In [ ]:
len(tissues)

In [ ]:
len(concatenated_gene_embed)

In [ ]:
concatenated_gene_embed

In [ ]:
# 4. Neighbors + UMAP
sc.pp.neighbors(adata, use_rep='X')
sc.tl.umap(adata)
sc.pl.umap(adata,
           size=3,
           color=label,
           show=False)
plt.title(f'{key} by {label} (UMAP)', fontsize=14)
plt.savefig(
    f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/figures/pan_tams_umap.png',
    dpi=300,
    bbox_inches='tight')
plt.clf()

# 5. PCA
sc.tl.pca(adata, use_highly_variable=False, svd_solver='arpack')
sc.pl.pca(adata,
          color=label,
          size=3,
          show=False)
plt.title(f'{key} by {label} (PCA)', fontsize=14)
plt.savefig(
    f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/figures/pan_tams_pca.png',
    dpi=300,
    bbox_inches='tight')
plt.clf()

In [ ]:
label = 'ensembl_id'

# 4. Neighbors + UMAP
sc.pp.neighbors(adata, use_rep='X')
sc.tl.umap(adata)
sc.pl.umap(adata,
           size=3,
           color=label,
           show=False)
plt.title(f'{key} by {label} (UMAP)', fontsize=14)
plt.savefig(
    f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/figures/pan_tams_umap.png',
    dpi=300,
    bbox_inches='tight')
plt.clf()

# 5. PCA
sc.tl.pca(adata, use_highly_variable=False, svd_solver='arpack')
sc.pl.pca(adata,
          color=label,
          size=3,
          show=False)
plt.title(f'{key} by {label} (PCA)', fontsize=14)
plt.savefig(
    f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/figures/pan_tams_pca.png',
    dpi=300,
    bbox_inches='tight')
plt.clf()

In [ ]:
 # 3. Create AnnData
adata = ad.AnnData(concatenated_gene_embed)
adata.obs['cell_id'] = [
    final_dataset['cell_id']] * len([k for k, v in output_gene_embed.items() if 'cell_emb' in k])
adata.obs['tissue'] = np.tile(
    filtered_tissues, len([k for k, v in output_gene_embed.items() if 'cell_emb' in k]))

# 4. Neighbors + UMAP
sc.pp.neighbors(adata, use_rep='X')
sc.tl.umap(adata)
sc.pl.umap(adata,
           size=3,
           color=label,
           show=False)
plt.title(f'{key} by {label} (UMAP)', fontsize=14)
plt.savefig(
    f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/figures/pan_tams_umap.png',
    dpi=300,
    bbox_inches='tight')
plt.clf()

# 5. PCA
sc.tl.pca(adata, use_highly_variable=False, svd_solver='arpack')
sc.pl.pca(adata,
          color=label,
          size=3,
          show=False)
plt.title(f'{key} by {label} (PCA)', fontsize=14)
plt.savefig(
    f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/figures/pan_tams_pca.png',
    dpi=300,
    bbox_inches='tight')
plt.clf()

In [ ]:
size = 1

In [ ]:
import os
import scanpy as sc

file_path_list = []
n_genes_list = []

for subdir, _, files in os.walk('/nfs/team361/sb75/DATASETS/silver'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad'):
            file_path = os.path.join(subdir, file)
            print(f'Loading AnnData with file path {file_path}.')
            adata = sc.read_h5ad(file_path)
            n_genes = len(adata.var)
            print(f'Number of genes: {n_genes}.')
            file_path_list.append(file_path)
            n_genes_list.append(n_genes)

In [ ]:
## 2.3 Analysis Data

In [ ]:
### 2.3.1 Kidney Only

In [ ]:
#### 2.3.1 Run Model Inference

In [ ]:
# Load token dict
with open(token_dict_path, 'rb') as f:
    token_dict = pickle.load(f)

token_dict_reverse =  {value: key for key, value in token_dict.items()}

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
                 '1007_batch0',
                 '1007_batch1',
                 '1007_batch2',
                 '1007_batch3',
                 '1007_batch4',
                 '1007_batch5',
                 '1007_batch6',
                 '1007_batch7',
                 '1007_batch8',
                 '1007_batch9',
                 '1007_batch10',
                 '1007_batch11',
                 '1007_batch12',
                 '1007_batch13',
                 '1007_batch14',
                 '1007_batch15',
                 '1007_batch16',
                 '1007_batch17',
                 '1007_batch18',
                 '1007_batch19',
                 '1007_batch20',
                 '1007_batch21',
                 '1007_batch22',
                 '1007_batch23',
                 '1007_batch24',
                 '1007_batch25',
                 '1007_batch26',
                 '1007_batch27',
                 '1007_batch28',
                 '1007_batch29',
                 '1007_batch30',
                 '1007_batch31',
                 '1007_batch32',
                 '1007_batch33',
                 '1007_batch34',
                 '1007_batch35',
                 '1007_batch36',
                 '1007_batch37',
                 '1007_batch38',
                 '1007_batch39',
                 '1007_batch40',]
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

In [ ]:
# Load data
_, _, dataset = prepare_dataset(params)
dataset

In [ ]:
# Subset dataset
step_size = 350000
steps = int(np.ceil(len(dataset)/step_size))

dataset_subsets = []

for i in range(steps):
    if ((i+1)*step_size) > len(dataset):
        end_step = len(dataset)
    else:
        end_step = (i+1)*step_size
    dataset_subsets.append(dataset.select(np.arange(i*step_size, end_step)))

del dataset

In [ ]:
# Get batches for current dataset subset
for i, dataset_subset in enumerate(dataset_subsets):
    print(f'Iteration: {i}')
    #if i != 1:
    #    continue
    
    # Get dataset IDs
    dataset_ids = ['1007']
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_1007_{step_size}_{i}.h5ad')
    del adata

In [ ]:
#### 2.3.2 Combine Data

In [ ]:
load_timestamp = '11062025_183320_055'
step_size = '350000'
read_checkpoint = 'ep10_0'
emb_key = 'neighborhood_emb_layer_6'

In [ ]:
adata_list = []

for i in range(8):
    adata = sc.read_h5ad(f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/adata_{read_checkpoint}_1007_{step_size}_{i}.h5ad')
    adata_list.append(adata)
    del adata

adata_full = sc.concat(adata_list, join='outer', index_unique=None)

In [ ]:
adata_full.write('adata_ep10_0_1007_complete.h5ad')

In [ ]:
!free -h

In [ ]:
### 2.3.2 Joint Skin, Kidney, Embryo

In [ ]:
# Load token dict
with open(token_dict_path, 'rb') as f:
    token_dict = pickle.load(f)

token_dict_reverse =  {value: key for key, value in token_dict.items()}

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
                 '1007_batch0',
                 '1007_batch1',
                 '1007_batch2',
                 '1007_batch3',
                 '1007_batch4',
                 '1007_batch5',
                 '1007_batch6',
                 '1007_batch7',
                 '1007_batch8',
                 '1007_batch9',
                 '1007_batch10',
                 '1007_batch11',
                 '1007_batch12',
                 '1007_batch13',
                 '1007_batch14',
                 '1007_batch15',
                 '1007_batch16',
                 '1007_batch17',
                 '1007_batch18',
                 '1007_batch19',
                 '1007_batch20',
                 '1007_batch21',
                 '1007_batch22',
                 '1007_batch23',
                 '1007_batch24',
                 '1007_batch25',
                 '1007_batch26',
                 '1007_batch27',
                 '1007_batch28',
                 '1007_batch29',
                 '1007_batch30',
                 '1007_batch31',
                 '1007_batch32',
                 '1007_batch33',
                 '1007_batch34',
                 '1007_batch35',
                 '1007_batch36',
                 '1007_batch37',
                 '1007_batch38',
                 '1007_batch39',
                 '1007_batch40',]
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
                 '1000_batch0',
                 '1000_batch1',
                 '1000_batch2',
                 '1000_batch3',
                 '1000_batch4',
                 '1000_batch5',
                 '1000_batch6',
                 '1000_batch7',
                 '1000_batch8',
                 '1000_batch9',
                 '1000_batch10',
                 '1000_batch11',
                 '1000_batch12',
                 '1000_batch13',
                 '1000_batch14',
                 '1000_batch15',
                 '1000_batch16',
                 '1000_batch17',
                 '1000_batch18',
                 '1000_batch19',
                 '1000_batch20',
                 '1000_batch21',
                 '1000_batch22',
                 '1000_batch23',
                 '1000_batch24',
                 '1000_batch25',
                 '1000_batch26',
                 '1000_batch27',
                 '1000_batch28',
                 '1000_batch29',
                 '1000_batch30',
                 '1000_batch31',
                 '1000_batch32',
                 '1000_batch33',
                 '1000_batch34',
                 '1000_batch35',
                 '1000_batch36',
                 '1000_batch37',
                 '1000_batch38',
                 '1007_batch0',
                 '1007_batch1',
                 '1007_batch2',
                 '1007_batch3',
                 '1007_batch4',
                 '1007_batch5',
                 '1007_batch6',
                 '1007_batch7',
                 '1007_batch8',
                 '1007_batch9',
                 '1007_batch10',
                 '1007_batch11',
                 '1007_batch12',
                 '1007_batch13',
                 '1007_batch14',
                 '1007_batch15',
                 '1007_batch16',
                 '1007_batch17',
                 '1007_batch18',
                 '1007_batch19',
                 '1007_batch20',
                 '1007_batch21',
                 '1007_batch22',
                 '1007_batch23',
                 '1007_batch24',
                 '1007_batch25',
                 '1007_batch26',
                 '1007_batch27',
                 '1007_batch28',
                 '1007_batch29',
                 '1007_batch30',
                 '1007_batch31',
                 '1007_batch32',
                 '1007_batch33',
                 '1007_batch34',
                 '1007_batch35',
                 '1007_batch36',
                 '1007_batch37',
                 '1007_batch38',
                 '1007_batch39',
                 '1007_batch40',
                 '1008_batch0',
                 '1008_batch1',
                 '1008_batch2',
                 '1008_batch3',
                 '1008_batch4',
                 '1008_batch5',
                 '1008_batch6',
                 '1008_batch7',
                 '1008_batch8',
                 '1008_batch9',
                 '1008_batch10',
                 '1008_batch11',
                 '1008_batch12',
                 '1008_batch13',
                 '1008_batch14',
                 '1008_batch15',
                 '1008_batch16',
                 '1008_batch17',
                 '1008_batch18',
                 '1008_batch19']
params['data']['val_batch_ids'] = None

In [ ]:
# Load data
_, _, dataset = prepare_dataset(params)
dataset

In [ ]:
dataset

In [ ]:
# Subset dataset
step_size = 350000
steps = int(np.ceil(len(dataset)/step_size))

dataset_subsets = []

for i in range(steps):
    if ((i+1)*step_size) > len(dataset):
        end_step = len(dataset)
    else:
        end_step = (i+1)*step_size
    dataset_subsets.append(dataset.select(np.arange(i*step_size, end_step)))

del dataset

In [ ]:
# Get batches for current dataset subset
for i, dataset_subset in enumerate(dataset_subsets):
    print(f'Iteration: {i}')
    if i != 0:
        continue
    
    # Get dataset IDs
    dataset_ids = ['1007']
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_1007.h5ad')
    del adata

In [ ]:
### 2.3.2 GBM

In [ ]:
# Load token dict
with open(token_dict_path, 'rb') as f:
    token_dict = pickle.load(f)

token_dict_reverse =  {value: key for key, value in token_dict.items()}

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [f'1002_batch{i}' for i in range(75)]
params['data']['val_batch_ids'] = None

In [ ]:
# Load data
_, _, dataset = prepare_dataset(params)
dataset

In [ ]:
# Subset dataset
step_size = 2000000
steps = int(np.ceil(len(dataset)/step_size))

dataset_subsets = []

for i in range(steps):
    if ((i+1)*step_size) > len(dataset):
        end_step = len(dataset)
    else:
        end_step = (i+1)*step_size
    dataset_subsets.append(dataset.select(np.arange(i*step_size, end_step)))

del dataset

In [ ]:
len(dataset_subsets)

In [ ]:
from pathlib import Path
folder_path = Path(f'/lustre/scratch126/cellgen/team361/DATASETS/silver/xhb1002-AT9-6b_1p')
adata_path_list = [p for p in folder_path.rglob("*")]

for adata_path in adata_path_list:
    adata = sc.read_h5ad(adata_path)
    print(f'{adata_path}: {adata.uns["batch"]}')

In [ ]:
# Get batches for current dataset subset
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 8:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_1002_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Run model inference
adata = infer(params,
              test_dataset,
              load_folder_path=folder_path,
              dataset_ids=list(set([batch_id.split('_')[0] for batch_id in params['data']['test_batch_ids']])),
              agg_type=agg_type,
              masked_tokens=None,
              agg_excluded_tokens=None,
              cell_gene_ids=[],
              neighborhood_gene_ids=[])

# Save adata
# adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

In [ ]:
# Set inference params
layer = params['meta']['enc_depth']
exclude_unshared_tokens = False # True, False
agg_type = 'avg' # 'avg', 'weighted_avg', 'cls'
latent_leiden_resolution = 0.5 # 0.1, 0.5
latent_cluster_key = f'leiden_res{str(latent_leiden_resolution).replace(".", "_")}'

In [ ]:
item = dataset[0]

In [ ]:
item['cell_id']

In [ ]:
dataset_subset = dataset.filter(lambda x: x['cell_id'].split('_')[0] in batch_ids)

In [ ]:
dataset_subset

In [ ]:
params['data']['tokenized_data_folder_path']

In [ ]:
test_subset = test_dataset.select(range(10))
test_subset
gene_ids = list(np.unique(test_subset['gene_tokens']))

# Run model inference
adata = infer(params,
              test_subset,
              load_folder_path=folder_path,
              dataset_ids=list(set([batch_id.split('_')[0] for batch_id in params['data']['test_batch_ids']])),
              obs_cols=[niche_key],
              agg_type='avg',
              masked_tokens=None,
              agg_excluded_tokens=None,
              cell_gene_ids=[],
              neighborhood_gene_ids=gene_ids)

# Save adata
#adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

norms = []

for i in range(10):
    for key in adata.obsm.keys():
        if 'neighborhood_emb_gene' in key:
            gene_emb = adata.obsm[key][i]
            norm = np.linalg.norm(gene_emb)
            norms.append(norm)

filtered_data = [x for x in norms if x != 0]

# Plot histogram of filtered data
plt.hist(filtered_data, bins=5, edgecolor='black')

In [ ]:
## 2.4 Miscellaneous

In [ ]:
# Check batches
from pathlib import Path
folder_path = Path(f'/lustre/scratch126/cellgen/team361/DATASETS/silver/xhb1002-AT9-6b_1p')
adata_path_list = [p for p in folder_path.rglob("*")]

for adata_path in adata_path_list:
    adata = sc.read_h5ad(adata_path)
    print(f'{adata_path}: {adata.uns["batch"]}')

In [ ]:
adata_batch = adata[adata.obs['slide_id'] == 'B2']

sc.pl.spatial(adata_batch,
              color='subregion',
              spot_size=30)

In [ ]:
sc.pp.calculate_qc_metrics(
    adata,
    inplace=True
)

import seaborn as sns
import matplotlib.pyplot as plt
sns.histplot(data=adata.obs, x='n_genes_by_counts', bins=100)
plt.xlabel('Number of expressed genes per cell')
plt.ylabel('Number of cells')
plt.show()

In [ ]:
# Pickle the list
with open("human_cohort2_60m_32_gene_corrected_read_depth_nonzero_mean_None_gene_corrected_read_depth_None_shifted_log_knn_10_train.pkl", "rb") as f:
    train = pickle.load(f)

with open("human_cohort2_60m_32_gene_corrected_read_depth_nonzero_mean_None_gene_corrected_read_depth_None_shifted_log_knn_10_test.pkl", "rb") as f:
    test = pickle.load(f)

with open("human_cohort2_60m_32_None_None_None_gene_corrected_read_depth_None_shifted_log_knn_10_train.pkl", "rb") as f:
    new_train = pickle.load(f)

with open("human_cohort2_60m_32_None_None_None_gene_corrected_read_depth_None_shifted_log_knn_10_test.pkl", "rb") as f:
    new_test = pickle.load(f)

print(len(train))
print(len(test))
print(len(new_train))
print(len(new_test))

train_matches = set(train).intersection(new_train)
print(len(train_matches))
test_matches = set(test).intersection(new_test)
print(len(test_matches))

train_differences = []
for i in range(len(train)):
    if train[i] != new_train[i]:
        train_differences.append((i, train[i], new_train[i]))
print(train_differences)
print(len(differences))

test_differences = []
for i in range(len(test)):
    if test[i] != new_test[i]:
        test_differences.append((i, test[i], new_test[i]))
print(test_differences[-10:])